# 15. The numbers of the paper

From the saved predictions only (`paper.py`): the main table (B = 10,000) and the curves by number of articles (B =
2,000) for every regressor and three baselines without a model of the text; paired comparisons with Holm correction
(main table and curves against TF-IDF; the claims of the text in one family); then the robustness analyses: notoriety
of the outlets (Tranco rank), political orientation (gap at equal score, the gap the compression alone would produce,
calibration by group), who is left out by the cleaning, and the leakage ladder of step 14.

In [ ]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())

import paper as PA

art, outlets, rows = PA.load()
models = PA.available()
print("models:", models)

## Main table

In [ ]:
T, V_main = PA.main_table(art, outlets, rows, models=models)
T.to_csv(C.OUT / "paper_main_table.csv", index=False)
T.assign(text=[f"{v:.3f} [{a:.3f}; {b:.3f}]" if m in ("Spearman", "Kendall") else f"{v:.1f} [{a:.1f}; {b:.1f}]"
               for v, a, b, m in zip(T.value, T.low, T.high, T.measure)]) \
 .pivot_table(index="model", columns="measure", values="text", aggfunc="first")[PA.MEASURES]

In [ ]:
T[T.p.notna()][["model", "measure", "diff", "dlo", "dhi", "p", "p_holm", "sig"]].round(4)

## Curves by number of articles

In [ ]:
Cv, V_curves = PA.curves(art, outlets, rows, models=models)
Cv.to_csv(C.OUT / "paper_curves.csv", index=False)
Cv.pivot_table(index="model", columns=["measure", "k"], values="value").round(3)[[("Spearman", k) for k in PA.K] + [("MAE", k) for k in PA.K]]

In [ ]:
Cv[Cv.p.notna()].pivot_table(index="model", columns=["measure", "k"], values="beats_tfidf", aggfunc="first")

## The claims of the text (one Holm family)

In [ ]:
Cl = PA.claims(V_main, V_curves, models)
Cl.to_csv(C.OUT / "paper_claims.csv", index=False)
Cl.round(4)

## Notoriety of the outlets (Tranco rank)

In [ ]:
f = C.DATA / "risorse" / "tranco" / "top-1m.csv"
if f.exists():
    N = PA.notoriety(art, outlets, rows)
    N.to_csv(C.OUT / "paper_notoriety.csv", index=False)
    display(N.round(3))
else:
    print("no Tranco list in", f)

## Political orientation

In [ ]:
O = PA.orientation(art, outlets, rows, models=models)
O.to_csv(C.OUT / "paper_orientation.csv", index=False)
O.assign(text=[f"{v:+.1f} [{a:+.1f}; {b:+.1f}]" for v, a, b in zip(O.value, O.low, O.high)]) \
 .pivot_table(index="model", columns="quantity", values="text", aggfunc="first")

## Who is left out by the cleaning

In [ ]:
f = C.OUT / "ladder" / "outlets_by_step.csv"
if f.exists():
    S, tests = PA.selection(pd.read_csv(f, index_col="domain"))
    S.to_csv(C.OUT / "paper_selection.csv", index=False)
    display(S.round(1)); print(tests)
else:
    print("run step 14 first")

## The leakage ladder (step 14)

In [ ]:
f = C.OUT / "paper_ladder.csv"
if f.exists():
    L = pd.read_csv(f)
    display(L.pivot_table(index="rung", columns=["measure", "model"], values="value").round(3))
else:
    print("run step 14 first")